# 🧬 Qiskit Classifier — Exploration Notebook

This notebook walks through the full pipeline of the `qiskit-classifier` project step by step:

1. Load and inspect the dataset
2. Visualise the data
3. Build and inspect the quantum circuits
4. Train the VQC classifier
5. Evaluate and visualise results

> **Prerequisites:** install the project with `pip install -e ".[dev,notebooks]"` from the repo root.

---
## 0. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

from qiskit_classifier.data import load_binary_iris
from qiskit_classifier.circuits import build_feature_map, build_ansatz
from qiskit_classifier.models import VQCClassifier
from qiskit_classifier.utils import plot_confusion_matrix, draw_circuit

# Reproducibility
RANDOM_STATE = 42
print('Imports OK ✓')

---
## 1. Load & Inspect the Dataset

We use a binary subset of the classic **Iris dataset** (classes 0 and 1 only).
Features are scaled to **[0, π]** to match the input range of quantum rotation gates.

In [ ]:
X_train, X_test, y_train, y_test = load_binary_iris(random_state=RANDOM_STATE)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Features         : {X_train.shape[1]}')
print(f'Classes          : {np.unique(y_train).tolist()}')
print(f'Feature range    : [{X_train.min():.4f}, {X_train.max():.4f}]  (expected [0, π ≈ 3.1416])')

In [ ]:
FEATURE_NAMES = ['Sepal length', 'Sepal width', 'Petal length', 'Petal width']
CLASS_NAMES   = ['Iris Setosa (0)', 'Iris Versicolor (1)']

print('── Training set ──')
for i, name in enumerate(FEATURE_NAMES):
    vals = X_train[:, i]
    print(f'  {name:<14}  mean={vals.mean():.3f}  std={vals.std():.3f}  min={vals.min():.3f}  max={vals.max():.3f}')

---
## 2. Visualise the Data

A pair-plot of all feature combinations helps us understand class separability.

In [ ]:
colors = ['#2196F3', '#FF5722']
markers = ['o', 's']

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
fig.suptitle('Pair Plot — Binary Iris (scaled to [0, π])', fontsize=14, y=1.01)

for row in range(4):
    for col in range(4):
        ax = axes[row, col]
        if row == col:
            # Diagonal: histogram per class
            for cls in [0, 1]:
                mask = y_train == cls
                ax.hist(X_train[mask, row], bins=12, alpha=0.6,
                        color=colors[cls], label=CLASS_NAMES[cls])
            ax.set_xlabel(FEATURE_NAMES[row], fontsize=8)
        else:
            for cls in [0, 1]:
                mask = y_train == cls
                ax.scatter(X_train[mask, col], X_train[mask, row],
                           c=colors[cls], marker=markers[cls],
                           alpha=0.6, s=20, label=CLASS_NAMES[cls])
            ax.set_xlabel(FEATURE_NAMES[col], fontsize=8)
            ax.set_ylabel(FEATURE_NAMES[row], fontsize=8)
        ax.tick_params(labelsize=7)

# Single shared legend
handles = [
    plt.Line2D([0],[0], marker=markers[i], color='w', markerfacecolor=colors[i],
               markersize=8, label=CLASS_NAMES[i])
    for i in range(2)
]
fig.legend(handles=handles, loc='upper right', fontsize=9, framealpha=0.9)
fig.tight_layout()
plt.show()

> **Observation:** Petal length and petal width (features 2 & 3) show the clearest
> class separation. Sepal width (feature 1) is the least discriminative.

---
## 3. Build & Inspect the Quantum Circuits

Before training, it's worth inspecting the two circuit components.

### 3.1 Feature Map (ZZFeatureMap)

Encodes classical input features into a quantum state.

In [ ]:
NUM_QUBITS = X_train.shape[1]  # 4

feature_map = build_feature_map(NUM_QUBITS, reps=2)
print(f'Feature map  — qubits: {feature_map.num_qubits}, parameters: {feature_map.num_parameters}')
feature_map.decompose().draw(output='mpl', style='clifford', fold=-1)

### 3.2 Ansatz (RealAmplitudes)

The trainable part of the circuit — its rotation angles are the model's "weights".

In [ ]:
ansatz = build_ansatz(NUM_QUBITS, reps=3)
print(f'Ansatz       — qubits: {ansatz.num_qubits}, trainable parameters: {ansatz.num_parameters}')
ansatz.decompose().draw(output='mpl', style='clifford', fold=-1)

### 3.3 Full Circuit (Feature Map + Ansatz combined)

In [ ]:
full_circuit = feature_map.compose(ansatz)
print(f'Full circuit — depth: {full_circuit.decompose().depth()}, total parameters: {full_circuit.num_parameters}')
full_circuit.decompose().draw(output='mpl', style='clifford', fold=-1)

---
## 4. Train the VQC Classifier

We instantiate `VQCClassifier` and call `.fit()` — just like any scikit-learn model.

> ⏱️ **Note:** Training may take a few minutes on a standard laptop (100 iterations × circuit evaluations).

In [ ]:
clf = VQCClassifier(
    num_qubits=NUM_QUBITS,
    feature_map_reps=2,
    ansatz_reps=3,
    max_iter=100,
)

print('Training VQC …')
clf.fit(X_train, y_train)
print('Training complete ✓')

---
## 5. Evaluate Results

### 5.1 Accuracy

In [ ]:
train_acc = clf.score(X_train, y_train)
test_acc  = clf.score(X_test,  y_test)

print(f'Train accuracy : {train_acc:.3f}  ({train_acc*100:.1f}%)')
print(f'Test  accuracy : {test_acc:.3f}  ({test_acc*100:.1f}%)')

### 5.2 Classification Report

In [ ]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

### 5.3 Confusion Matrix

In [ ]:
fig = plot_confusion_matrix(y_test, y_pred, labels=CLASS_NAMES)
plt.show()

### 5.4 Per-sample Prediction Breakdown

In [ ]:
correct   = np.sum(y_pred == y_test)
incorrect = np.sum(y_pred != y_test)

fig, ax = plt.subplots(figsize=(8, 3))
x = np.arange(len(y_test))

for i in x:
    color = '#4CAF50' if y_pred[i] == y_test[i] else '#F44336'
    ax.bar(i, 1, color=color, width=0.8)

ax.set_xticks(x)
ax.set_xticklabels([f'{i}' for i in x], fontsize=7)
ax.set_yticks([])
ax.set_xlabel('Test sample index')
ax.set_title(f'Predictions on test set — ✓ {correct} correct  ✗ {incorrect} wrong')

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#4CAF50', label='Correct'),
    Patch(color='#F44336', label='Incorrect'),
], loc='upper right')

fig.tight_layout()
plt.show()

---
## 6. Experiment: Effect of `ansatz_reps`

More repetitions = more parameters = higher model capacity, but also slower training.
Let's compare test accuracy for `reps ∈ {1, 2, 3}`.

> ⏱️ This cell trains 3 models — it may take several minutes.

In [ ]:
results = {}

for reps in [1, 2, 3]:
    model = VQCClassifier(num_qubits=NUM_QUBITS, ansatz_reps=reps, max_iter=80)
    model.fit(X_train, y_train)
    acc = model.score(X_test, y_test)
    results[reps] = acc
    print(f'  ansatz_reps={reps}  →  test accuracy={acc:.3f}')

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar([str(k) for k in results], results.values(), color='#5C6BC0', width=0.5)
ax.set_xlabel('ansatz_reps')
ax.set_ylabel('Test accuracy')
ax.set_ylim(0, 1.05)
ax.set_title('Test accuracy vs ansatz repetitions')
ax.axhline(1.0, color='grey', linestyle='--', linewidth=0.8)
fig.tight_layout()
plt.show()

---
## 7. Sklearn Integration: Cross-Validation

Because `VQCClassifier` is sklearn-compatible, standard tools like `cross_val_score` work out of the box.

> ⏱️ This trains 5 models (one per fold) — expect a longer wait.

In [ ]:
from sklearn.model_selection import cross_val_score
import numpy as np

# Combine train+test for cross-validation
X_all = np.vstack([X_train, X_test])
y_all = np.concatenate([y_train, y_test])

cv_model = VQCClassifier(num_qubits=NUM_QUBITS, max_iter=80)
scores = cross_val_score(cv_model, X_all, y_all, cv=5, scoring='accuracy')

print(f'CV scores : {[f"{s:.3f}" for s in scores]}')
print(f'Mean ± std: {scores.mean():.3f} ± {scores.std():.3f}')

---
## Summary

| Step | What we did |
|------|-------------|
| Data | Loaded binary Iris, scaled features to [0, π] |
| Circuits | Inspected ZZFeatureMap (encoder) and RealAmplitudes (ansatz) |
| Training | Fitted VQCClassifier with L-BFGS-B optimiser |
| Evaluation | Accuracy, classification report, confusion matrix |
| Experiments | Effect of `ansatz_reps` on accuracy; 5-fold cross-validation |

**Next steps to try:**
- Swap `ZZFeatureMap` for `PauliFeatureMap` in `circuits/feature_map.py`
- Try `COBYLA` or `SPSA` optimisers (better for noisy hardware)
- Load your own CSV dataset and plug it into the same pipeline